# EveryMind 988 Training Survey: From Raw Export to Memo Findings

This notebook is the team's single technical artifact. It takes the survey's raw Excel export, cleans it the same way every time, and produces every number, table and chart used in the process memo.

It is organized to follow the memo:

| Memo section | Notebook section |
|---|---|
| Methodology | 1. Load and clean, 2. Data quality report |
| Key finding: where the training works | 3. Did the training work, and for whom? |
| Key finding: Spanish speakers are undercounted | 4. Spanish speakers and survey access |
| Key finding: people leave with unanswered questions | 5. Unanswered questions |
| EveryMind's questions (younger people, communities of color, intent) | 6. Supporting findings |
| Recommendations | 7. Evidence behind each recommendation |
| Implementation | 8. Quarterly refresh and exports |
| Limitations | 9. Limitations, computed from the data |
| (all of the above) | 10. Every number the memo cites |

**A note on AI and charts.** AI tools helped write this code. Every chart is printed next to the table it was drawn from, so the numbers can be checked by eye, and the memo quotes the tables, not the charts.

## How to run

1. Put the survey's Excel export in the same folder as this notebook (or in a `data/` folder next to it). The default file name is `buildfest_raw.xlsx`; change `RAW_FILE_NAME` below if yours is different.
2. Optional: put `acs_2024_5yr_md_counties.csv` (Census county figures) in the same place to get the reach comparison in section 6.
3. Optional: put `names_to_redact.txt` (one presenter or staff name per line) in the same place. Names in comments are replaced with `[name]`. Keep this file private; it is not part of the shared artifact.
4. Run all cells (Jupyter: *Run All*; Google Colab: *Runtime > Run all*). Results are saved to the `output/` folder.

The notebook needs `pandas`, `numpy`, `matplotlib` and `openpyxl` (all preinstalled in Google Colab).

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)


RAW_FILE_NAME = "buildfest_raw.xlsx"                  # the survey's Excel export
RAW_SHEET = "Uncleaned Survey Data"                   # used if present, otherwise the first sheet
CENSUS_FILE_NAME = "acs_2024_5yr_md_counties.csv"     # optional
NAMES_FILE_NAME = "names_to_redact.txt"               # optional, private
OUT_DIR = Path("output")
MIN_GROUP_N = 20          # groups smaller than this are marked "small group" in every table
TOO_FAST_SECONDS = 20     # responses finished faster than this are flagged


SEARCH_FOLDERS = [Path("."), Path("data"), Path("/content"), Path("/content/data"), Path("/content/drive/MyDrive")]

def find_file(name, required=True):
    """Return the path to `name` in the usual folders, or None if optional and missing.
    Usage: find_file("buildfest_raw.xlsx")"""
    for folder in SEARCH_FOLDERS:
        p = folder / name
        if p.exists():
            return p
    if required:
        raise FileNotFoundError(
            f"Could not find '{name}'. Put it in the same folder as this notebook or in a data/ folder, "
            f"or change the file name setting above.")
    return None

RAW_PATH = find_file(RAW_FILE_NAME)
CENSUS_PATH = find_file(CENSUS_FILE_NAME, required=False)
NAMES_PATH = find_file(NAMES_FILE_NAME, required=False)
OUT_DIR.mkdir(exist_ok=True)

print("Survey export :", RAW_PATH)
print("Census file   :", CENSUS_PATH or "not found (section 6 reach table will be skipped)")
print("Names file    :", NAMES_PATH or "not found (only names written as 'Ms. X', 'Mr. X' etc. are removed)")

ModuleNotFoundError: No module named 'numpy'

---
## 1. Load and clean (memo: Methodology)

Every step is a small function, so the same cleaning runs on every new export. Each step **flags** problems instead of silently dropping rows, and anything unexpected in a new export (a new answer label, a new county spelling, a missing question) is printed as a warning so staff notice it.

### 1.1 Load the export and give columns short names
Columns are matched by the wording of the question, not by position, so the notebook still works if the survey tool reorders columns or adds a new one.

In [ ]:
def norm_text(x):
    """Lowercase, remove hidden characters, collapse spaces. Usage: norm_text("Strongly agree\xa0")"""
    x = str(x).replace("\xa0", " ")
    for ch in ["‎", "‏", "​", "﻿"]:
        x = x.replace(ch, "")
    return re.sub(r"\s+", " ", x).strip().lower()

# short name -> pattern found in the original question text
COLUMN_RULES = [
    ("row_id",             r"^(power query|id)$"),
    ("training_date",      r"date of your training"),
    ("time_taken",         r"^time taken"),
    ("date_taken",         r"^date taken"),
    ("start_time",         r"^start time"),
    ("completion_time",    r"^(trim\(|completion time)"),
    ("language",           r"^language"),
    ("role_raw",           r"are you a"),
    ("county_raw",         r"which county"),
    ("post_identify",      r"please tell us.*identify"),
    ("post_describe",      r"please tell us.*describe 988"),
    ("post_when_use",      r"please tell us.*when to use"),
    ("pre_identify",       r"think back.*identify"),
    ("pre_describe",       r"think back.*describe 988"),
    ("pre_when_use",       r"think back.*when to use"),
    ("presenter_engaged",  r"engaged the participants"),
    ("presenter_knew",     r"knew the subject"),
    ("well_organized",     r"well organized"),
    ("recommend_0_10",     r"likely are you to recommend"),
    ("likely_call_0_10",   r"likely are you to call"),
    ("most_valuable_text", r"most valuable"),
    ("improve_text",       r"improve this training"),
]
REQUIRED = ["role_raw", "county_raw", "language", "post_identify", "post_describe", "post_when_use",
            "pre_identify", "pre_describe", "pre_when_use", "recommend_0_10"]

def load_export(path, sheet):
    """Read the export and rename columns using COLUMN_RULES. Returns (table, codebook)."""
    sheets = pd.ExcelFile(path).sheet_names
    use = sheet if sheet in sheets else sheets[0]
    raw = pd.read_excel(path, sheet_name=use)
    mapping, unmatched = {}, []
    for col in raw.columns:
        key = norm_text(col)
        hit = next((new for new, pat in COLUMN_RULES if re.search(pat, key) and new not in mapping.values()), None)
        if hit:
            mapping[col] = hit
        else:
            unmatched.append(col)
    missing = [c for c in REQUIRED if c not in mapping.values()]
    if missing:
        raise ValueError(f"The export is missing required questions: {missing}. Check the survey form or COLUMN_RULES.")
    if unmatched:
        warnings.warn(f"New or unknown columns kept as-is (not analyzed): {unmatched}")
    df = raw.rename(columns=mapping)
    for new, _ in COLUMN_RULES:          # optional questions that are absent become blank columns
        if new not in df.columns:
            df[new] = np.nan
    codebook = pd.DataFrame({"short_name": list(mapping.values()),
                             "original_question": [re.sub(r"\s+", " ", str(c).replace("\xa0", " ")).strip() for c in mapping]})
    print(f"Read sheet '{use}': {raw.shape[0]} responses, {raw.shape[1]} columns")
    return df, codebook

df, codebook = load_export(RAW_PATH, RAW_SHEET)
N_RAW = len(df)
codebook

NameError: name 'RAW_PATH' is not defined

### 1.2 Fix formatting: hidden characters, IDs, dates, language
Answers like `"Strongly agree\xa0"` (an invisible trailing space) count as a different answer until cleaned.

In [ ]:
def clean_cell(x):
    """Remove hidden characters and extra spaces from a text cell; empty text becomes blank."""
    if not isinstance(x, str):
        return x
    for bad in ["\xa0"]:
        x = x.replace(bad, " ")
    for bad in ["‎", "‏", "​", "﻿"]:
        x = x.replace(bad, "")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else np.nan

for c in df.columns:
    if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
        df[c] = df[c].map(clean_cell)

df["row_id"] = range(1, len(df) + 1)                       # first row had a blank ID in the original export
for c in ["training_date", "date_taken"]:
    df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()
for c in ["start_time", "completion_time"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["seconds_taken"] = pd.to_timedelta(df["time_taken"].astype(str), errors="coerce").dt.total_seconds()
df["seconds_taken"] = df["seconds_taken"].fillna((df["completion_time"] - df["start_time"]).dt.total_seconds())

def form_language(x):
    """'English (United States)' -> 'English', 'Español (México)' -> 'Spanish'."""
    x = norm_text(x)
    if x.startswith("english"):
        return "English"
    if x.startswith("espa") or x.startswith("spanish"):
        return "Spanish"
    return "Other" if x not in ("", "nan") else np.nan

df["form_language"] = df["language"].map(form_language)
print(df["form_language"].value_counts(dropna=False).to_string())

### 1.3 Counties, roles and agree/disagree answers
* Counties are typed many ways (`Montgomery County with NAMI MC`, `Fairfax county`). They are mapped to one name each; out-of-state rows (Fairfax VA, Harris TX) are flagged and left out of the analysis.
* The role question allows several answers, so it becomes one yes/no column per role plus a single `role_primary`.
* Agree/disagree answers become 1 (Strongly disagree) to 5 (Strongly agree). `Very agree` and `Satisfied` (from an older form version) are mapped to Strongly agree and Agree and flagged.

In [ ]:
MD_COUNTIES = ["Allegany", "Anne Arundel", "Baltimore", "Baltimore City", "Calvert", "Caroline", "Carroll", "Cecil",
               "Charles", "Dorchester", "Frederick", "Garrett", "Harford", "Howard", "Kent", "Montgomery",
               "Prince George's", "Queen Anne's", "St. Mary's", "Somerset", "Talbot", "Washington", "Wicomico", "Worcester"]
OTHER_COUNTIES = {"fairfax": ("Fairfax", "VA"), "harris": ("Harris", "TX")}
COUNTY_LOOKUP = {c.lower().replace("’", "'"): (c, "MD") for c in MD_COUNTIES} | OTHER_COUNTIES

def standardize_county(x):
    """Map messy county text to (county, state). Unknown text is kept and its state is 'Unknown'."""
    if not isinstance(x, str):
        return (np.nan, "Unknown")
    k = re.sub(r"\bcounty\b.*$", "", x.lower().replace("’", "'")).strip()
    return COUNTY_LOOKUP.get(k, (x, "Unknown"))

pairs = df["county_raw"].map(standardize_county)
df["county"] = pairs.str[0]
df["state"] = pairs.str[1]
df["in_md_service_area"] = df["state"].eq("MD")
unknown_counties = sorted(df.loc[df["state"].eq("Unknown"), "county_raw"].dropna().unique())
if unknown_counties:
    warnings.warn(f"County spellings not recognized (left out of the analysis until added to COUNTY_LOOKUP): {unknown_counties}")

ROLE_PATTERNS = {"is_student": r"student", "is_school_staff": r"school staff",
                 "is_caregiver": r"caregiver|parent", "is_community_member": r"other|community"}
ROLE_LABELS = {"is_student": "Student", "is_school_staff": "School staff",
               "is_caregiver": "Caregiver/parent", "is_community_member": "Community member"}
role_text = df["role_raw"].fillna("").str.lower()
for col, pat in ROLE_PATTERNS.items():
    df[col] = role_text.str.contains(pat, regex=True)
def primary_role(row):
    """The single role picked, 'Multiple roles', or blank if none."""
    picked = [ROLE_LABELS[c] for c in ROLE_PATTERNS if row[c]]
    return np.nan if not picked else (picked[0] if len(picked) == 1 else "Multiple roles")

df["role_primary"] = df.apply(primary_role, axis=1)
df["age_group"] = np.where(df["is_student"], "Students", "Adults")

PREPOST = ["post_identify", "post_describe", "post_when_use", "pre_identify", "pre_describe", "pre_when_use"]
ASPECTS = ["presenter_engaged", "presenter_knew", "well_organized"]
LIKERT_COLS = PREPOST + ASPECTS
LIKERT_FIX = {"neither agree or disagree": "Neither agree nor disagree", "very agree": "Strongly agree", "satisfied": "Agree"}
LIKERT_SCORE = {"strongly disagree": 1, "disagree": 2, "neither agree nor disagree": 3, "agree": 4, "strongly agree": 5}

df["had_nonstandard_likert"] = df[LIKERT_COLS].apply(lambda s: s.map(norm_text)).isin(["very agree", "satisfied"]).any(axis=1)
unknown_labels = set()
for c in LIKERT_COLS:
    df[c] = df[c].map(lambda v: LIKERT_FIX.get(norm_text(v), v) if isinstance(v, str) else v)
    df[c + "_num"] = df[c].map(lambda v: LIKERT_SCORE.get(norm_text(v)) if isinstance(v, str) else np.nan).astype(float)
    unknown_labels |= set(df.loc[df[c].notna() & df[c + "_num"].isna(), c])
if unknown_labels:
    warnings.warn(f"Agree/disagree labels not recognized (treated as blank): {sorted(unknown_labels)}")

for c in ["recommend_0_10", "likely_call_0_10"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df.loc[~df[c].between(0, 10), c] = np.nan
df["asked_likely_call"] = df["likely_call_0_10"].notna()

print(df["county"].value_counts(dropna=False).to_string(), "\n")
print(df["role_primary"].value_counts(dropna=False).to_string())

### 1.4 Spanish speakers (memo finding 2)
The survey records which **form** was shown, not the language people speak. Some people on the English form wrote their comments in Spanish, so we also detect the language of the written answers by counting common Spanish and English words. A respondent counts as a Spanish speaker if either the form or the comments are in Spanish.

In [ ]:
ES_WORDS = set("""que para los las una por con del como pero muy mas más hay ayuda saber esta está son mis nuestros
nuestras hijos hija hijo familia gracias información informacion puede personas cuando donde todo todos siempre
llamar apoyo yo tengo nada bien sobre también tambien cómo qué algo ellos entrenamiento""".split())
EN_WORDS = set("""the and to is you that of it can how for this with are when what call about know learned someone
people was have be not if or more""".split())

def detect_language(text):
    """'Spanish' or 'English' for one written answer, by word counts. Usage: detect_language("que hay ayuda")"""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    words = re.findall(r"[a-záéíóúñü]+", text.lower())
    es = sum(w in ES_WORDS for w in words)
    en = sum(w in EN_WORDS for w in words)
    if re.search(r"[ñ¿¡]", text.lower()) or (es >= 2 and es > en):
        return "Spanish"
    return "English"

both_text = (df["most_valuable_text"].fillna("") + " " + df["improve_text"].fillna("")).str.strip()
df["text_language"] = both_text.map(detect_language).fillna("No text")
df["spanish_speaker"] = df["form_language"].eq("Spanish") | df["text_language"].eq("Spanish")
pd.crosstab(df["form_language"], df["text_language"], margins=True)

### 1.5 Remove names from comments
First-round judging is blind and comments sometimes name presenters. Names in the private `names_to_redact.txt` file, and any name written with a title (`Ms. Morales`), become `[name]`.

In [ ]:
NAMES = []
if NAMES_PATH:
    NAMES = [n.strip() for n in NAMES_PATH.read_text(encoding="utf-8").splitlines() if n.strip()]
NAMES.sort(key=len, reverse=True)                     # longer names first ("Ann Lee" before "Ann")
TITLED_NAME = r"\b(?:Mr|Mrs|Ms|Miss|Dr|Coach)\.?\s+[A-Z][a-z'’-]+"

def redact_names(text):
    """Replace listed names and titled names with [name]. Non-text is returned unchanged."""
    if not isinstance(text, str):
        return text
    for n in NAMES:
        text = re.sub(rf"\b{re.escape(n)}\b", "[name]", text, flags=re.IGNORECASE)
    return re.sub(TITLED_NAME, "[name]", text)

n_redacted = 0
for c in ["most_valuable_text", "improve_text"]:
    before = df[c].copy()
    df[c] = df[c].map(redact_names)
    n_redacted += int((before.fillna("") != df[c].fillna("")).sum())
print("comments with a name removed:", n_redacted)

### 1.6 Quality flags and the final analysis set
Rows are flagged, not deleted. The analysis set keeps responses that:
* answered all six before/after questions,
* are in the Maryland service area,
* are **not** both too fast *and* the same answer throughout (either one alone is kept),
* are **not** a likely reversed-scale answer (all three "after" answers are Strongly disagree, yet the training was rated highly),
* are **not** a duplicate submission (identical long comments).

Other questions are analyzed only on rows that answered them; a blank is never treated as a low score.

In [ ]:
POST_NUM = ["post_identify_num", "post_describe_num", "post_when_use_num"]
PRE_NUM = ["pre_identify_num", "pre_describe_num", "pre_when_use_num"]
ASPECT_NUM = [c + "_num" for c in ASPECTS]

df["aspects_avg"] = df[ASPECT_NUM].mean(axis=1)
df["flag_too_fast"] = df["seconds_taken"] < TOO_FAST_SECONDS
df["flag_straightline"] = df[PREPOST].nunique(axis=1).eq(1) & df[ASPECTS].nunique(axis=1).eq(1)
df["flag_training_after_survey"] = df["training_date"] > df["date_taken"]
df["flag_reversed_scale"] = (df[POST_NUM].eq(1).all(axis=1)
                             & ((df["aspects_avg"] >= 4) | (df["recommend_0_10"] >= 7)))

def comment_key(row):
    """Both comments joined, lowercased, or None if either is short. Short answers repeat naturally."""
    v, i = row["most_valuable_text"], row["improve_text"]
    if not (isinstance(v, str) and isinstance(i, str)) or min(len(v), len(i)) < 20:
        return None
    return v.lower().strip(" .") + " || " + i.lower().strip(" .")

keys = df.apply(comment_key, axis=1)
df["flag_possible_duplicate"] = keys.notna() & keys.duplicated(keep="first")
df["complete_core"] = df[[c + "_num" for c in PREPOST]].notna().all(axis=1)
junk = df["flag_too_fast"] & df["flag_straightline"]
df["in_analysis"] = (df["complete_core"] & df["in_md_service_area"] & ~junk
                     & ~df["flag_reversed_scale"] & ~df["flag_possible_duplicate"])

# derived measures used throughout
for it in ["identify", "describe", "when_use"]:
    df[f"gain_{it}"] = df[f"post_{it}_num"] - df[f"pre_{it}_num"]
df["gain_avg"] = df[["gain_identify", "gain_describe", "gain_when_use"]].mean(axis=1)
df["confident_post"] = df[POST_NUM].ge(4).all(axis=1)
df["had_room_to_grow"] = df[PRE_NUM].lt(4).any(axis=1)
df["at_ceiling_pre"] = df[PRE_NUM].eq(5).all(axis=1)
df["quarter"] = df["date_taken"].dt.to_period("Q").astype(str)

a = df[df["in_analysis"]].copy()       # the analysis set used in every section below
N = len(a)
print(f"{N_RAW} responses in the export -> {N} in the analysis set")

---
## 2. Data quality report (memo: Methodology)
Counts are computed, not typed, so the report stays correct on new exports.

In [ ]:
quality = pd.DataFrame([
    ("Responses in the export", N_RAW),
    ("Out-of-state or unknown county (left out)", int((~df["in_md_service_area"]).sum())),
    ("Missing a before/after answer (left out)", int((~df["complete_core"]).sum())),
    ("Finished in under %d seconds (flagged)" % TOO_FAST_SECONDS, int(df["flag_too_fast"].sum())),
    ("Same answer to every question (flagged)", int(df["flag_straightline"].sum())),
    ("Too fast AND same answer throughout (left out)", int(junk.sum())),
    ("Likely read the scale backwards (left out)", int(df["flag_reversed_scale"].sum())),
    ("Duplicate submission (left out)", int(df["flag_possible_duplicate"].sum())),
    ("Training date after survey date (flagged)", int(df["flag_training_after_survey"].sum())),
    ("'Very agree' / 'Satisfied' recoded (flagged)", int(df["had_nonstandard_likert"].sum())),
    ("Comments with a name removed", n_redacted),
    ("Responses in the analysis set", N),
], columns=["step", "rows"])
quality

---
## 3. Did the training work, and for whom? (memo: Recommendations, "Automate the review")

"Agree" means Agree or Strongly agree. *Before* answers were given **after** the session, looking back (a retrospective rating), which can make gains look a little bigger; see section 9.

In [ ]:
SKILLS = {"identify": "Spot a mental health crisis", "describe": "Explain what 988 is", "when_use": "Know when to use 988"}

def pct_agree(s):
    """Percent of non-blank 1-5 scores that are 4 or 5."""
    s = s.dropna()
    return round(100 * (s >= 4).mean(), 1) if len(s) else np.nan

def agree_table(data, by=None, min_n=MIN_GROUP_N):
    """Percent agree before and after for each skill, per group, with n and a small-group marker.
    Usage: agree_table(a) or agree_table(a, "role_primary")"""
    groups = [("Everyone", data)] if by is None else list(data.groupby(by))
    rows = []
    for name, g in groups:
        row = {"group": name, "n": len(g)}
        for k in SKILLS:
            row[f"{k}_before"] = pct_agree(g[f"pre_{k}_num"])
            row[f"{k}_after"] = pct_agree(g[f"post_{k}_num"])
            row[f"{k}_change_pts"] = round(row[f"{k}_after"] - row[f"{k}_before"], 1)
        row["pct_already_at_top"] = round(100 * g["at_ceiling_pre"].mean(), 1)
        row["small_group"] = len(g) < min_n
        rows.append(row)
    return pd.DataFrame(rows).set_index("group")

overall = agree_table(a)
overall

In [ ]:
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"

def before_after_chart(tab, title, path):
    """Dumbbell chart of % agree before vs after for each skill, from a one-row agree_table."""
    r = tab.iloc[0]
    labels = list(SKILLS.values())
    before = [r[f"{k}_before"] for k in SKILLS]
    after = [r[f"{k}_after"] for k in SKILLS]
    fig, ax = plt.subplots(figsize=(7, 3))
    y = np.arange(len(labels))
    ax.hlines(y, before, after, color="#d4d3cd", lw=3)
    ax.scatter(before, y, s=80, color=ORANGE, label="Before training (looking back)", zorder=3)
    ax.scatter(after, y, s=80, color=BLUE, label="After training", zorder=3)
    for i, (b, f) in enumerate(zip(before, after)):
        ax.text(b - 2, i, f"{b:.0f}%", va="center", ha="right", color=MUTED, fontsize=9)
        ax.text(f + 2, i, f"{f:.0f}%", va="center", color=INK, fontsize=9)
    ax.set_yticks(y, labels)
    ax.set_xlim(0, 105)
    ax.set_xlabel("% who agree", color=MUTED)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, -0.25), ncol=2, fontsize=9)
    ax.set_title(title, loc="left", color=INK)
    plt.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()

before_after_chart(overall, f"Confidence before vs after the 988 training (n = {N})", OUT_DIR / "chart_before_after.png")

**Read it like this:** the biggest jump is in explaining what 988 is; spotting a crisis moved least, partly because most people already agreed before. The same table by role is below. Every group is shown with its size, and groups under 20 people are marked so nobody over-reads them.

In [ ]:
by_role = agree_table(a, "role_primary")
by_role

**The average hides the impact for the people the training is for.** About a third of respondents already agreed with all three statements before the session and cannot show much gain. Splitting them out:

In [ ]:
room = (a.groupby("had_room_to_grow")
          .agg(n=("row_id", "size"), avg_gain_1_to_5=("gain_avg", "mean"), pct_confident_after=("confident_post", "mean"))
          .rename(index={True: "Had room to grow", False: "Already confident before"}))
room["pct_of_people"] = 100 * room["n"] / room["n"].sum()
room["pct_confident_after"] *= 100
room.round(2)

---
## 4. Spanish speakers and survey access (memo key finding 1)

In [ ]:
spanish = pd.Series({
    "Used the Spanish form": int(a["form_language"].eq("Spanish").sum()),
    "Wrote comments in Spanish": int(a["text_language"].eq("Spanish").sum()),
    "Wrote in Spanish on the English form": int((a["form_language"].eq("English") & a["text_language"].eq("Spanish")).sum()),
    "Spanish speakers (form or comments)": int(a["spanish_speaker"].sum()),
}).to_frame("respondents")
spanish["pct_of_analysis_set"] = (100 * spanish["respondents"] / N).round(1)
spanish

**Did the training work as well for Spanish speakers?** Compare outcomes, and read them with the `pct_already_at_top` column: many Spanish speakers already chose the top answer before training, which caps how much they can gain. With fewer than about 100 Spanish speakers, a difference of a few points can come from a handful of answers.

In [ ]:
by_spanish = agree_table(a.assign(group=np.where(a["spanish_speaker"], "Spanish speakers", "Everyone else")), "group")
by_spanish

In [ ]:
sp = by_spanish
describe_gap = sp.loc["Everyone else", "describe_change_pts"] - sp.loc["Spanish speakers", "describe_change_pts"]
after_gap = sp.loc["Everyone else", "describe_after"] - sp.loc["Spanish speakers", "describe_after"]
print(f"'Explain 988' after training: Spanish speakers {sp.loc['Spanish speakers','describe_after']}% vs everyone else {sp.loc['Everyone else','describe_after']}% ({after_gap:+.1f} pts)")
print(f"'Explain 988' change: Spanish speakers {sp.loc['Spanish speakers','describe_change_pts']} pts vs everyone else {sp.loc['Everyone else','describe_change_pts']} pts")
print(f"Already at the top before training: Spanish speakers {sp.loc['Spanish speakers','pct_already_at_top']}% vs {sp.loc['Everyone else','pct_already_at_top']}%")
print("\nMemo wording to use: compare AFTER-training levels (where both groups end up), and note the ceiling effect;"
      " do not claim equal gains if the change column differs by more than a few points.")

---
## 5. Unanswered questions (memo key finding 2)

When asked *"How can we improve this training?"*, many people wrote a question instead of a suggestion. This section measures how many, what they asked about, and whether that share changes over time.

**How to read the numbers:**
* The base is **everyone who wrote anything in the improvement box**, including answers like "nothing" or "it was great". It is not "people who suggested improvements".
* A comment counts as a question if it contains `?` (or `¿`) or starts like one (*how, what, why, can, does...*).
* Writing a question shows what people still wanted to know; it does not prove they were confused. The memo says "wrote a question instead of a suggestion", which is what the data shows.
* Topics are matched by keywords (mostly English), so a sample of each topic is printed below for a manual check.

In [ ]:
QUESTION_START = r"^(how|what|why|when|where|who|which|can|could|do|does|did|is|are|will|would|should|i wonder|i was wondering|i want to know|cómo|qué|por qué|puedo|se puede)\b"
TOPICS = {
    "Does 988 work / how many people use it":
        r"how many|how often|percent|%|statistic|numbers?\b|does (?:it|988) (?:really |actually )?work|effective|success|how much|data|cuánt|cuant",
    "What happens when you call":
        r"what happens|who (?:answers|picks up|is on|will)|what (?:do|will|would|does) (?:they|the person|someone)|what to expect|after (?:you|i|we) call|police|911|trace|track|wait|qué pasa|que pasa",
    "Can you contact 988 if not in crisis / for someone else":
        r"not in (?:a )?crisis|not (?:in )?an emergency|someone else|a friend|for others|another person|if (?:i|you)'?(?:m| am| are)? not|small|bad day|for (?:anything|any reason)|amig|famil|otra persona",
}

imp = a.loc[a["improve_text"].notna(), ["row_id", "role_primary", "improve_text"]].copy()
t = imp["improve_text"].str.lower()
imp["is_question"] = t.str.contains(r"[?¿]") | t.str.match(QUESTION_START)
for topic, pat in TOPICS.items():
    imp[topic] = imp["is_question"] & t.str.contains(pat, regex=True)

n_improve = len(imp)
n_questions = int(imp["is_question"].sum())
questions = pd.DataFrame({
    "comments": [n_improve, n_questions] + [int(imp[tp].sum()) for tp in TOPICS],
}, index=["Answered 'how can we improve'", "Wrote a question"] + list(TOPICS))
questions["pct_of_improve_answers"] = (100 * questions["comments"] / n_improve).round(1)
questions

**What else is in the improvement box?** Splitting every answer into questions, "no change needed / praise", and real suggestions shows how much of the box is actually improvement ideas.

In [ ]:
NO_CHANGE = r"^(?:none|no|nothing|n/?a|idk|nope|not sure|i don'?t know|it was (?:great|good|fine|awesome|perfect)|nada|todo bien|everything was (?:great|good|fine)|keep (?:it|doing))\\b|^(?:it'?s|it was)? ?(?:good|great|fine|perfect|awesome)[.! ]*$"
imp["answer_type"] = np.select(
    [imp["is_question"], t.str.contains(NO_CHANGE, regex=True)],
    ["Question", "No change needed / praise"], default="Suggestion")
box = imp["answer_type"].value_counts().reindex(["Suggestion", "Question", "No change needed / praise"], fill_value=0).to_frame("comments")
box["pct_of_improve_answers"] = (100 * box["comments"] / n_improve).round(1)
box

In [ ]:
# Topics ranked by how often they come up: the memo should list them in this order
topic_rank = questions.loc[list(TOPICS)].sort_values("comments", ascending=False)
print("Question topics, most common first:")
for i, (tp, r) in enumerate(topic_rank.iterrows(), 1):
    print(f"  {i}. {tp}: {int(r['comments'])} comments ({r['pct_of_improve_answers']}% of improvement answers)")
print(f"Questions that fit none of the three topics: {int((imp['is_question'] & ~imp[list(TOPICS)].any(axis=1)).sum())}")

# Consistency check for the memo's earlier figure (222 of 651)
both_blank = int((a["most_valuable_text"].isna() & a["improve_text"].isna()).sum())
print(f"\nCheck: respondents who left BOTH comment boxes blank = {both_blank}.")
print("If the memo's question count equals this number, it was likely copied from the wrong table;"
      f" use the question count above ({n_questions}) instead.")

In [ ]:
# manual check: read a sample from each topic; adjust TOPICS if comments are mis-sorted
for topic in TOPICS:
    sample = imp.loc[imp[topic], "improve_text"]
    print(f"\n=== {topic} ({len(sample)}) ===")
    print(sample.sample(min(8, len(sample)), random_state=1).to_string(index=False) if len(sample) else "(none)")
print("\n=== Questions in no topic (sample) ===")
other = imp.loc[imp["is_question"] & ~imp[list(TOPICS)].any(axis=1), "improve_text"]
print(other.sample(min(8, len(other)), random_state=1).to_string(index=False) if len(other) else "(none)")

---
## 6. Supporting findings for EveryMind's questions

EveryMind asked about younger people, communities of color, behavioral intent, and whether the training reaches its audience. These tables back the memo's recommendations and limitations.

### 6.1 Younger participants: learning vs. enjoying the session
"Learned" = agrees with all three statements after training. "Liked" = would recommend it (7 to 10). Students who learned the content but would not recommend it point at **delivery** (length, format), not content.

In [ ]:
def learn_vs_like(data):
    """Percent of respondents in each learned/liked box (rows with a recommend score only)."""
    d = data.dropna(subset=["recommend_0_10"])
    learned, liked = d["confident_post"], d["recommend_0_10"] >= 7
    return pd.Series({"n": len(d),
                      "learned and liked": 100 * (learned & liked).mean(),
                      "learned, would NOT recommend": 100 * (learned & ~liked).mean(),
                      "did not learn": 100 * (~learned).mean()}).round(1)

younger = pd.DataFrame({g: learn_vs_like(a[a["age_group"] == g]) for g in ["Students", "Adults"]}).T
younger

### 6.2 Behavioral intent: would they contact 988?
*"How likely are you to call 988?"* was only on some versions of the form, so this uses only people who were asked.

In [ ]:
intent = (a[a["asked_likely_call"]].groupby("age_group")
            .agg(n=("row_id", "size"),
                 avg_0_to_10=("likely_call_0_10", "mean"),
                 pct_9_or_10=("likely_call_0_10", lambda s: 100 * (s >= 9).mean()),
                 pct_6_or_below=("likely_call_0_10", lambda s: 100 * (s <= 6).mean()))
            .round(1))
intent["small_group"] = intent["n"] < MIN_GROUP_N
print(f"Asked the question: {int(a['asked_likely_call'].sum())} of {N} ({100*a['asked_likely_call'].mean():.0f}%)")
intent

### 6.3 Reach: who attends vs. who lives in each county
The survey has no race, ethnicity or age question, so communities of color can only be approached through **county** demographics (Census ACS 2020 to 2024 estimates). This is a county-level comparison; it says nothing about individual respondents.

In [ ]:
reach = (a.groupby("county")
           .agg(responses=("row_id", "size"),
                students_trained=("is_student", "sum"),
                pct_respondents_spanish=("spanish_speaker", lambda s: round(100 * s.mean(), 1)))
           .reset_index())
if CENSUS_PATH:
    acs = pd.read_csv(CENSUS_PATH)
    reach = acs.merge(reach, on="county", how="left").fillna({"responses": 0, "students_trained": 0})
    reach["students_trained_per_1000_youth"] = (1000 * reach["students_trained"] / reach["youth_12_17"]).round(1)
    keep = ["county", "population", "pct_people_of_color", "pct_spanish_at_home", "responses",
            "pct_respondents_spanish", "youth_12_17", "students_trained", "students_trained_per_1000_youth"]
    reach = reach[[c for c in keep if c in reach.columns]]
else:
    print("Census file not found: showing survey counts only.")
reach["small_group"] = reach["responses"] < MIN_GROUP_N
reach

In [ ]:
if CENSUS_PATH and "students_trained_per_1000_youth" in reach:
    r = reach.sort_values("students_trained_per_1000_youth")
    fig, ax = plt.subplots(figsize=(7, 2.8))
    ax.barh(r["county"], r["students_trained_per_1000_youth"], color=BLUE, height=0.55)
    for i, (v, poc) in enumerate(zip(r["students_trained_per_1000_youth"], r["pct_people_of_color"])):
        ax.text(v + 0.3, i, f"{v:.1f}   (county {poc:.0f}% people of color)", va="center", fontsize=8, color=INK)
    ax.set_xlabel("Students trained per 1,000 residents aged 12 to 17", color=MUTED)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("Youth reach by county", loc="left", color=INK)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "chart_youth_reach.png", dpi=200, bbox_inches="tight")
    plt.show()

---
## 7. Evidence behind each recommendation (memo: Recommendations)
Each recommendation, the number from this notebook that supports it, and the measure EveryMind would track to know it worked. The table rebuilds itself on each run.

In [ ]:
def _get(tab, row, col):
    try:
        return tab.loc[row, col]
    except KeyError:
        return np.nan

recs = pd.DataFrame([
    ("1. Automate the quarterly review (this notebook)",
     f"Clean, flagged results for {N} of {N_RAW} responses in one run",
     "Same tables every quarter (section 8)"),
    ("2. Language choice at the start of the survey",
     f"{int(spanish.loc['Wrote in Spanish on the English form','respondents'])} people wrote in Spanish on the English form; "
     f"only {int(spanish.loc['Used the Spanish form','respondents'])} used the Spanish form",
     "Spanish-form share rises toward the Spanish-comment share"),
    ("3. Answer the questions people leave with: in-session Q&A and a walk-through of what happens when you contact 988; "
     "a separate optional 'What questions do you still have?' survey box answered in a follow-up FAQ; "
     "the most common questions turned into short quiz items",
     f"{questions.loc['Wrote a question','pct_of_improve_answers']}% of people who answered the improvement question wrote a question "
     f"instead of a suggestion; most common topic: {topic_rank.index[0]}",
     "Fewer people ask these questions each quarter (counted across the improvement box and the new questions box); "
     "at least 80% answer the quiz items correctly"),
    ("4. Student version of the session (shorter, interactive)",
     f"{_get(younger,'Students','learned, would NOT recommend')}% of students learned it but would not recommend it "
     f"(adults {_get(younger,'Adults','learned, would NOT recommend')}%)",
     "'Learned, would not recommend' share for students falls"),
    ("5. Ask 'call, text, or chat' on every form",
     f"Students average {_get(intent,'Students','avg_0_to_10')}/10 likely to call; only "
     f"{100*a['asked_likely_call'].mean():.0f}% of forms asked",
     "Intent measured on every response"),
], columns=["recommendation", "evidence", "how to know it worked"])
recs

---
## 8. Quarterly refresh and exports (memo: Implementation)

**Who:** one staff member. **How often:** once a quarter (plus a quick look at the first few responses after the form changes). **What they need:** the survey's Excel export and this notebook (Google Colab works with no setup). **Time:** about one to two hours including team review.

The table below tracks the main outcomes by quarter so trends are visible, and the cell after it saves everything to `output/` for the team review or a dashboard.

In [ ]:
def by_quarter(data):
    """Outcomes per survey quarter, split by students and adults (never one combined score)."""
    rows = []
    for (q, g), d in data.groupby(["quarter", "age_group"]):
        rows.append({"quarter": q, "group": g, "n": len(d),
                     "pct_confident_after": round(100 * d["confident_post"].mean(), 1),
                     "describe_988_after": pct_agree(d["post_describe_num"]),
                     "avg_recommend": round(d["recommend_0_10"].mean(), 2),
                     "n_asked_likely_call": int(d["asked_likely_call"].sum()),
                     "pct_improve_answers_that_are_questions": round(100 * imp.loc[imp["row_id"].isin(d["row_id"]), "is_question"].mean(), 1)
                         if imp["row_id"].isin(d["row_id"]).any() else np.nan,
                     "small_group": len(d) < MIN_GROUP_N})
    return pd.DataFrame(rows)

quarterly = by_quarter(a)
quarterly

Students and adults are always reported separately. Most of the year's late sessions were student-only, so a single combined score would fall even if every group scored the same as before.

In [ ]:
CLEAN_COLS = (["row_id", "training_date", "date_taken", "quarter", "seconds_taken", "county", "role_primary", "age_group",
               "is_student", "is_school_staff", "is_caregiver", "is_community_member",
               "form_language", "text_language", "spanish_speaker"]
              + PRE_NUM + POST_NUM + ["gain_identify", "gain_describe", "gain_when_use", "gain_avg",
              "confident_post", "had_room_to_grow"] + ASPECT_NUM
              + ["aspects_avg", "recommend_0_10", "likely_call_0_10", "asked_likely_call",
                 "most_valuable_text", "improve_text", "had_nonstandard_likert", "flag_too_fast", "flag_straightline",
                 "flag_training_after_survey"])

results_path = OUT_DIR / "results.xlsx"
with pd.ExcelWriter(results_path, engine="openpyxl") as xw:
    for name, frame in [("1_overall", overall), ("2_by_role", by_role), ("3_room_to_grow", room),
                        ("4_spanish", spanish), ("5_spanish_outcomes", by_spanish), ("6_questions", questions), ("6b_improve_box", box),
                        ("7_students_vs_adults", younger), ("8_intent", intent), ("9_reach", reach),
                        ("10_by_quarter", quarterly), ("11_recommendations", recs), ("12_data_quality", quality)]:
        frame.to_excel(xw, sheet_name=name)
    a[CLEAN_COLS].to_excel(xw, sheet_name="clean_data", index=False)
    df.to_excel(xw, sheet_name="all_rows_with_flags", index=False)
    codebook.to_excel(xw, sheet_name="codebook", index=False)
a[CLEAN_COLS].to_csv(OUT_DIR / "clean_data.csv", index=False)
print("Saved:", results_path, "and", OUT_DIR / "clean_data.csv", "and charts in", OUT_DIR)

---
## 9. Limitations, computed from the data (memo: Limitations)

In [ ]:
limits = pd.DataFrame([
    ("Results mostly describe one group",
     f"{100*a['is_student'].mean():.0f}% of respondents are students and "
     f"{100*a['county'].eq('Anne Arundel').mean():.0f}% are from Anne Arundel County."),
    ("'Before' answers are retrospective",
     "People rated their 'before' knowledge after the session, from memory. This is a standard method but tends to inflate gains."),
    ("Intent is measured on a small subset",
     f"'Likely to call' was on only {100*a['asked_likely_call'].mean():.0f}% of forms ({int(a['asked_likely_call'].sum())} people), "
     "and it asks only about calling, not texting or chatting."),
    ("Some training objectives are not measured",
     "The survey has no items on risk factors or protective factors; warning signs and how 988 works are covered only through self-rated confidence."),
    ("Communities of color cannot be measured directly",
     "There is no race, ethnicity or age question; county demographics are a community-level proxy only."),
    ("Language is estimated",
     f"Spanish speakers ({int(a['spanish_speaker'].sum())}) are detected from the form or written comments; bilingual people who wrote in English are missed."),
    ("Only respondents are counted",
     "Without attendance numbers we cannot tell how many attendees did not respond or how they differ."),
    ("Keyword coding is approximate",
     "Question topics (section 5) use keyword rules checked by reading samples, not a full manual coding; Spanish questions may be under-counted."),
    ("A question is not proof of confusion",
     "People who wrote a question in the improvement box may have been curious rather than confused; the finding is what they asked, not what they misunderstood."),
], columns=["limitation", "detail"])
limits

---
## 10. Every number the memo cites
Copy these into the memo after each run so the memo and the artifact always match.

In [ ]:
o = overall.iloc[0]
memo_numbers = {
    "Responses in the export": N_RAW,
    "Responses analyzed": N,
    "Explain 988: % agree before -> after": f"{o['describe_before']:.0f}% -> {o['describe_after']:.0f}%",
    "Spot a crisis: % agree before -> after": f"{o['identify_before']:.0f}% -> {o['identify_after']:.0f}%",
    "Know when to use 988: % agree before -> after": f"{o['when_use_before']:.0f}% -> {o['when_use_after']:.0f}%",
    "Used the Spanish form": f"{spanish.loc['Used the Spanish form','respondents']} ({spanish.loc['Used the Spanish form','pct_of_analysis_set']}%)",
    "Wrote comments in Spanish": f"{spanish.loc['Wrote comments in Spanish','respondents']} ({spanish.loc['Wrote comments in Spanish','pct_of_analysis_set']}%)",
    "  ...of those, on the English form": spanish.loc["Wrote in Spanish on the English form", "respondents"],
    "Spanish speakers (either)": f"{spanish.loc['Spanish speakers (form or comments)','respondents']} ({spanish.loc['Spanish speakers (form or comments)','pct_of_analysis_set']}%)",
    "Explain 988 after: Spanish speakers vs others": f"{sp.loc['Spanish speakers','describe_after']}% vs {sp.loc['Everyone else','describe_after']}%",
    "Answered 'how can we improve'": n_improve,
    "Wrote a question instead of a suggestion": f"{n_questions} ({100*n_questions/n_improve:.0f}%)",
    "Wrote a real suggestion": f"{box.loc['Suggestion','comments']} ({box.loc['Suggestion','pct_of_improve_answers']}%)",
    **{f"  Question topic #{i}: {tp}": int(r["comments"]) for i, (tp, r) in enumerate(topic_rank.iterrows(), 1)},
    "Students: learned but would not recommend": f"{_get(younger,'Students','learned, would NOT recommend')}%",
    "Students: avg likely to call (0-10)": _get(intent, "Students", "avg_0_to_10"),
    "% students / % Anne Arundel": f"{100*a['is_student'].mean():.0f}% / {100*a['county'].eq('Anne Arundel').mean():.0f}%",
}
pd.Series(memo_numbers, name="value").to_frame()